# 06 - Model Explainability (SHAP)

## Objective
Explain the selected model at two levels: globally (which features drive the book)
and individually (why *this* applicant received *this* probability). Every number
below comes from computed SHAP values -- nothing is written by hand.

SHAP values are additive contributions in log-odds space: positive pushes the
applicant towards default, negative away from it.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src import config
from src.data import preprocessing
from src.features.engineering import add_credit_risk_features
from src.models import explain as E
from src.models import train as T

%matplotlib inline

## Configuration and data loading

In [ ]:
SAMPLE_SIZE = 2000
raw = preprocessing.load_raw_data()
TARGET = preprocessing.detect_target(raw)
clean, _ = preprocessing.clean_data(raw, TARGET)
df = add_credit_risk_features(clean)
splits = T.split_data(df, TARGET)
model = T.load_model()          # saved by run_pipeline.py / notebook 05
model.named_steps["classifier"]

## Building the explainer

In [ ]:
sample = splits["X_test"].sample(min(SAMPLE_SIZE, len(splits["X_test"])),
                                 random_state=config.RANDOM_STATE)
explainer, transformed = E.build_explainer(model, sample)
shap_values = E.compute_shap_values(explainer, transformed)
shap_values.shape

## Global feature importance

In [ ]:
importance = E.global_importance(shap_values, transformed)
importance.head(15)

In [ ]:
E.save_summary_plots(shap_values, transformed)
import shap
shap.summary_plot(shap_values, transformed, max_display=15)

The beeswarm shows both magnitude and direction: for each feature, points to the
right increase predicted risk, and the colour shows whether that came from a high
or a low feature value.

In [ ]:
shap.summary_plot(shap_values, transformed, plot_type="bar", max_display=15)

## Individual prediction explanations

In [ ]:
scored_sample = sample.copy()
scored_sample["pd"] = model.predict_proba(sample)[:, 1]
riskiest = scored_sample["pd"].idxmax()
safest = scored_sample["pd"].idxmin()
scored_sample.loc[[riskiest, safest], ["person_income", "loan_amnt", "loan_grade",
                                       "loan_int_rate", "loan_percent_income", "pd"]]

In [ ]:
for label, index in [("HIGHEST predicted risk", riskiest), ("LOWEST predicted risk", safest)]:
    customer = sample.loc[[index]]
    explanation = E.explain_customer(model, explainer, customer)
    print(f"=== {label} | PD = {model.predict_proba(customer)[0, 1]:.4f} ===")
    for line in E.narrate_contributions(explanation):
        print(" -", line)
    print()

In [ ]:
explanation = E.explain_customer(model, explainer, sample.loc[[riskiest]], top_n=8)
pd.DataFrame(explanation["increases_risk"])

In [ ]:
pd.DataFrame(explanation["decreases_risk"])

## Contribution chart for a single applicant

In [ ]:
import matplotlib.pyplot as plt

contributions = (pd.DataFrame(explanation["increases_risk"] + explanation["decreases_risk"])
                 .drop_duplicates("feature")
                 .sort_values("shap_value"))
colours = ["#C0392B" if value > 0 else "#2E8B57" for value in contributions["shap_value"]]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(contributions["feature"], contributions["shap_value"], color=colours)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("SHAP value (log-odds contribution to default)")
ax.set_title(f"Drivers for the highest-risk applicant "
             f"(PD = {model.predict_proba(sample.loc[[riskiest]])[0, 1]:.3f})")
fig.tight_layout()

## Results
The ranked table and the plots above are the actual SHAP output; `run_pipeline.py`
writes the same ranking to `reports/shap_global_importance.csv`.

## Interpretation
Explanations are stated in terms of the transformed feature space (one-hot columns
included), which is what the model actually consumes. When reporting back to a
credit officer, translate each contribution into the applicant's own attribute --
for example, a large positive contribution on the loan-to-income ratio means the
requested amount is high relative to income.

## Caveat
SHAP explains the *model*, not the world. These are associations learned from
historical decisions, not causal effects, and `loan_grade` / `loan_int_rate` carry
the lender's prior risk assessment. Any production use needs fairness testing
across protected groups and human review of adverse decisions.

## Conclusion
The model's drivers match the EDA and the SQL analytics, so its behaviour is
defensible to a risk committee -- the final requirement before serving it through
the API and dashboard.